# Entrega 1: Visualizacion sobre mayores contaminantes en los Paises

### **Profesor:** Alessio bellino
### **Estudiantes:** Ninoska Cid, Lucas Chicao e Isidora Gomez

---

## **Libreria y carga de datos**

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Esta es la línea mágica que fuerza que se abra en el navegador
pio.renderers.default = 'browser'

df = pd.read_csv('Data/global air pollution dataset.csv', encoding='utf-8')
df.head()

,Country,City,AQI Value,AQI Category,CO AQI Value,CO AQI Category,Ozone AQI Value,Ozone AQI Category,NO2 AQI Value,NO2 AQI Category,PM2.5 AQI Value,PM2.5 AQI Category
0,Russian Federation,Praskoveya,51,Moderate,1,Good,36,Good,0,Good,51,Moderate
1,Brazil,Presidente Dutra,41,Good,1,Good,5,Good,1,Good,41,Good
2,Italy,Priolo Gargallo,66,Moderate,1,Good,39,Good,2,Good,66,Moderate
3,Poland,Przasnysz,34,Good,1,Good,34,Good,0,Good,20,Good
4,France,Punaauia,22,Good,0,Good,22,Good,0,Good,6,Good


In [2]:
#revision de datos 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23463 entries, 0 to 23462
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Country             23036 non-null  object
 1   City                23462 non-null  object
 2   AQI Value           23463 non-null  int64 
 3   AQI Category        23463 non-null  object
 4   CO AQI Value        23463 non-null  int64 
 5   CO AQI Category     23463 non-null  object
 6   Ozone AQI Value     23463 non-null  int64 
 7   Ozone AQI Category  23463 non-null  object
 8   NO2 AQI Value       23463 non-null  int64 
 9   NO2 AQI Category    23463 non-null  object
 10  PM2.5 AQI Value     23463 non-null  int64 
 11  PM2.5 AQI Category  23463 non-null  object
dtypes: int64(5), object(7)
memory usage: 2.1+ MB


In [3]:
#elimino los datos nulos que no mencionen Paises o Ciudades
df_clean = df.dropna(subset=['Country', 'City'])

## **Graficos**

In [4]:
# Grafico de los 7 Paises con Peor Calidad del aire
#separo los primeros 7 paises con peor calidad
top = df_clean.groupby('Country')['AQI Value'].mean().nlargest(7).reset_index().sort_values('AQI Value')

#veo el grafico con los paises que elegimos y un color que no desaparezca
fig = px.bar(top, x='AQI Value', y='Country', orientation='h', title='Top 7 Paises con Peor Calidad de Aire',color='AQI Value', color_continuous_scale='Greens')

#ajusto para que no se pierda vision en el grafico
fig.update_layout(bargap=0.4, height=500)

#muestro el grafico
fig.show()

In [5]:
# Grafico con el promedio de contaminantes
#separo los datos que voy a necesitar
promedios = df_clean[['PM2.5 AQI Value', 'Ozone AQI Value', 'NO2 AQI Value', 'CO AQI Value']].mean().reset_index()
promedios.columns = ['Contaminante', 'Promedio']

#cambio los nombres para que sean mas entendibles para el publico
nombres_faciles = {
    'PM2.5 AQI Value': 'Humo, Polvo y Hollín (PM2.5)',
    'Ozone AQI Value': 'Gas Ozono (Irritante)',
    'NO2 AQI Value': 'Gases de escape de autos',
    'CO AQI Value': 'Monóxido de Carbono'
}

promedios['Contaminante'] = promedios['Contaminante'].map(nombres_faciles)

#creo el grafico con los datps
fig = px.pie(promedios, values='Promedio', names='Contaminante', hole=0.4, title='¿Qué es lo que más ensucia el aire a nivel mundial?', color_discrete_sequence=px.colors.sequential.Blues_r)
fig.show()

# ----------------------------


In [6]:
# esta es otra forma

resumen = (
    df_clean.groupby('Country')
    .agg(
        pm25_prom   = ('PM2.5 AQI Value',  'mean'),
        ozono_prom  = ('Ozone AQI Value',  'mean'),
        num_ciudades = ('City', 'count')
    )
    .reset_index()   
)

resumen['contaminacion_prom'] = (
    resumen['pm25_prom'] + resumen['ozono_prom']
) / 2

resumen_top = resumen.nlargest(15, 'contaminacion_prom')

fig = px.scatter(
    resumen_top,
    x='pm25_prom',
    y='ozono_prom',
    size='num_ciudades',
    color='contaminacion_prom',
    hover_name='Country',
    title='Top 15 países con mayor contaminación promedio segun PM2.5 y Ozono<br><sup>Tamaño = cantidad de ciudades</sup>',
    labels={
        'pm25_prom': 'PM2.5 promedio',
        'ozono_prom': 'Ozono promedio',
        'num_ciudades': 'Número de ciudades',
        'contaminacion_prom': 'Contaminación promedio'
    },
    text='Country',
    size_max=200,
    
)
fig.update_traces(textposition='top center')
#fig.update_yaxes(range=[0, 200])
#fig.update_xaxes(range=[50, 250])

fig.show()